# OpenQASM Interop

qdk-pythonic can export and import both Q# and OpenQASM 3.0. This notebook demonstrates
round-trip conversion between formats.

In [ ]:
from qdk_pythonic import Circuit

## Build a Circuit and Export to OpenQASM

In [ ]:
circ = Circuit()
q = circ.allocate(2)
circ.h(q[0]).cx(q[0], q[1]).measure_all()

qasm_str = circ.to_openqasm()
print(qasm_str)

## Import an OpenQASM String

Use `Circuit.from_openqasm()` to parse an OpenQASM 3.0 source string back into a `Circuit` object.

In [ ]:
qasm_source = """
OPENQASM 3.0;
include "stdgates.inc";

qubit[3] q;

h q[0];
cx q[0], q[1];
cx q[1], q[2];
"""

imported = Circuit.from_openqasm(qasm_source)
print(imported.draw())
print(f"Qubits: {imported.qubit_count()}, Gates: {imported.gate_count()}")

## Round-Trip: OpenQASM -> Circuit -> Q# -> Circuit -> OpenQASM

Demonstrate a full round-trip through both code generation backends.

In [ ]:
# Start from an OpenQASM string
original_qasm = """
OPENQASM 3.0;
include "stdgates.inc";

qubit[2] q;

h q[0];
cx q[0], q[1];
"""

# OpenQASM -> Circuit
circ_from_qasm = Circuit.from_openqasm(original_qasm)
print("Step 1 - Imported from OpenQASM:")
print(circ_from_qasm.draw())
print()

# Circuit -> Q#
qsharp_str = circ_from_qasm.to_qsharp()
print("Step 2 - Exported to Q#:")
print(qsharp_str)
print()

# Q# -> Circuit
circ_from_qs = Circuit.from_qsharp(qsharp_str)
print("Step 3 - Re-imported from Q#:")
print(circ_from_qs.draw())
print()

# Circuit -> OpenQASM
final_qasm = circ_from_qs.to_openqasm()
print("Step 4 - Exported back to OpenQASM:")
print(final_qasm)

## Import from Q# Source

`Circuit.from_qsharp()` parses a Q# operation into a `Circuit`.

In [ ]:
qs_source = """
operation BellState() : Result[] {
    use qs = Qubit[2];
    H(qs[0]);
    CNOT(qs[0], qs[1]);
    return [M(qs[0]), M(qs[1])];
}
"""

circ_qs = Circuit.from_qsharp(qs_source)
print(circ_qs.draw())
print(f"Qubits: {circ_qs.qubit_count()}, Gates: {circ_qs.gate_count()}")

## JSON Serialization

Circuits can also be serialized to and from JSON for storage or transport.

In [ ]:
json_str = circ_qs.to_json()
print(json_str[:200], "...")

restored = Circuit.from_json(json_str)
print(f"\nRestored circuit: {restored.qubit_count()} qubits, {restored.gate_count()} gates")